<div style="
    background-color: white;
    padding: 22px;
    border-left: 8px solid #811433;
    border-radius: 12px;
    box-shadow: 0 4px 14px rgba(0,0,0,0.08);
    margin: 10px 0 20px 0;">
    <h1 style="
        color: #811433;
        margin: 0;
        font-family: Arial, sans-serif;
        font-size: 30px;">
        TP – Data Visualisation et Analyse Prédictive des Données Hospitalières
    </h1>
    <p style="
        color: #444;
        margin-top: 10px;
        font-size: 15px;
        font-family: Arial, sans-serif;">
        Exploration, visualisation, modélisation prédictive et interprétation métier
    </p>
</div>

<div style="
    background-color: white;
    padding: 14px 18px;
    border-radius: 10px;
    border: 2px solid #811433;
    margin: 18px 0 12px 0;">
    <h2 style="
        color: #811433;
        margin: 0;
        font-family: Arial, sans-serif;">
        1. Imports et configuration
    </h2>
</div>

In [1]:
# ================================
# 1. IMPORTS ET CONFIGURATION
# ================================

# Manipulation de données
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Prétraitement & Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Modèles de régression
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# Évaluation
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# Options d'affichage
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Style graphique
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

print("Imports chargés avec succès.")

Imports chargés avec succès.


In [2]:
# ================================
# 2. CHARGEMENT DES DONNÉES
# ================================

file_path = "../data/hospital_data.csv"

df = pd.read_csv(file_path, sep=";")

print("Dimensions du dataset :", df.shape)
display(df.head())

Dimensions du dataset : (500, 10)


,PatientID,Age,Sexe,Departement,Maladie,DureeSejour,Cout,DateAdmission,DateSortie,Traitement
0,1,82,M,Cardiologie,Cancer,5,2125,03/09/2025,08/09/2025,Chirurgie
1,2,18,M,Oncologie,Cancer,15,8685,16/01/2025,31/01/2025,Médication
2,3,76,F,Cardiologie,Infarctus,2,822,17/05/2025,19/05/2025,Chirurgie
3,4,65,M,Neurologie,Fracture,12,7584,08/06/2025,20/06/2025,Physiothérapie
4,5,70,F,Orthopédie,Alzheimer,10,4420,01/10/2025,11/10/2025,Médication


In [3]:
# ================================
# 3. STRUCTURE DE LA BASE
# ================================

print("Informations générales :")
display(df.info())

print("\nTypes de variables :")
display(df.dtypes)

print("\nValeurs manquantes par colonne :")
display(df.isnull().sum())

print("\nNombre de doublons :", df.duplicated().sum())

Informations générales :
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   PatientID      500 non-null    int64
 1   Age            500 non-null    int64
 2   Sexe           500 non-null    str  
 3   Departement    500 non-null    str  
 4   Maladie        500 non-null    str  
 5   DureeSejour    500 non-null    int64
 6   Cout           500 non-null    int64
 7   DateAdmission  500 non-null    str  
 8   DateSortie     500 non-null    str  
 9   Traitement     500 non-null    str  
dtypes: int64(4), str(6)
memory usage: 39.2 KB


None


Types de variables :


PatientID        int64
Age              int64
Sexe               str
Departement        str
Maladie            str
DureeSejour      int64
Cout             int64
DateAdmission      str
DateSortie         str
Traitement         str
dtype: object


Valeurs manquantes par colonne :


PatientID        0
Age              0
Sexe             0
Departement      0
Maladie          0
DureeSejour      0
Cout             0
DateAdmission    0
DateSortie       0
Traitement       0
dtype: int64


Nombre de doublons : 0


In [4]:
# ================================
# 4. NETTOYAGE ET PRÉPARATION
# ================================

# Copie de travail
data = df.copy()

# Conversion des dates
data["DateAdmission"] = pd.to_datetime(data["DateAdmission"], format="%d/%m/%Y", errors="coerce")
data["DateSortie"] = pd.to_datetime(data["DateSortie"], format="%d/%m/%Y", errors="coerce")

# Vérification cohérence durée calculée à partir des dates
data["DureeCalculee"] = (data["DateSortie"] - data["DateAdmission"]).dt.days

# Vérifier si la durée calculée correspond à la variable DureeSejour
data["EcartDuree"] = data["DureeSejour"] - data["DureeCalculee"]

print("Valeurs manquantes après conversion des dates :")
display(data[["DateAdmission", "DateSortie"]].isnull().sum())

print("\nAperçu des données préparées :")
display(data.head())

print("\nStatistiques descriptives des variables numériques :")
display(data[["Age", "DureeSejour", "Cout", "DureeCalculee", "EcartDuree"]].describe())

Valeurs manquantes après conversion des dates :


DateAdmission    0
DateSortie       0
dtype: int64


Aperçu des données préparées :


,PatientID,Age,Sexe,Departement,Maladie,DureeSejour,Cout,DateAdmission,DateSortie,Traitement,DureeCalculee,EcartDuree
0,1,82,M,Cardiologie,Cancer,5,2125,2025-09-03,2025-09-08,Chirurgie,5,0
1,2,18,M,Oncologie,Cancer,15,8685,2025-01-16,2025-01-31,Médication,15,0
2,3,76,F,Cardiologie,Infarctus,2,822,2025-05-17,2025-05-19,Chirurgie,2,0
3,4,65,M,Neurologie,Fracture,12,7584,2025-06-08,2025-06-20,Physiothérapie,12,0
4,5,70,F,Orthopédie,Alzheimer,10,4420,2025-10-01,2025-10-11,Médication,10,0



Statistiques descriptives des variables numériques :


,Age,DureeSejour,Cout,DureeCalculee,EcartDuree
count,500.00,500.00,500.00,500.00,500.00
mean,47.18,7.95,"4,060.12",7.95,0.00
std,25.05,4.21,"2,418.81",4.21,0.00
min,1.00,1.00,303.00,1.00,0.00
25%,27.00,5.00,"2,136.25",5.00,0.00
50%,49.00,8.00,"3,769.50",8.00,0.00
75%,69.00,12.00,"5,749.50",12.00,0.00
max,90.00,15.00,"10,380.00",15.00,0.00


In [5]:
# ================================
# 5. ENRICHISSEMENT DES VARIABLES
# ================================

# Mois d'admission
data["MoisAdmission"] = data["DateAdmission"].dt.month
data["NomMoisAdmission"] = data["DateAdmission"].dt.month_name()

# Jour de semaine d'admission
data["JourAdmission"] = data["DateAdmission"].dt.day_name()

# Tranches d'âge
data["TrancheAge"] = pd.cut(
    data["Age"],
    bins=[0, 18, 35, 50, 65, 80, 100],
    labels=["0-18", "19-35", "36-50", "51-65", "66-80", "81+"]
)

# Coût journalier
data["CoutParJour"] = data["Cout"] / data["DureeSejour"]

# Séjour long (utile pour la classification)
moyenne_sejour = data["DureeSejour"].mean()
data["SejourLong"] = (data["DureeSejour"] > moyenne_sejour).astype(int)

print("Durée moyenne de séjour :", round(moyenne_sejour, 2), "jours")
display(data.head())

Durée moyenne de séjour : 7.95 jours


,PatientID,Age,Sexe,Departement,Maladie,DureeSejour,Cout,DateAdmission,DateSortie,Traitement,DureeCalculee,EcartDuree,MoisAdmission,NomMoisAdmission,JourAdmission,TrancheAge,CoutParJour,SejourLong
0,1,82,M,Cardiologie,Cancer,5,2125,2025-09-03,2025-09-08,Chirurgie,5,0,9,September,Wednesday,81+,425.00,0
1,2,18,M,Oncologie,Cancer,15,8685,2025-01-16,2025-01-31,Médication,15,0,1,January,Thursday,0-18,579.00,1
2,3,76,F,Cardiologie,Infarctus,2,822,2025-05-17,2025-05-19,Chirurgie,2,0,5,May,Saturday,66-80,411.00,0
3,4,65,M,Neurologie,Fracture,12,7584,2025-06-08,2025-06-20,Physiothérapie,12,0,6,June,Sunday,51-65,632.00,1
4,5,70,F,Orthopédie,Alzheimer,10,4420,2025-10-01,2025-10-11,Médication,10,0,10,October,Wednesday,66-80,442.00,1


<div style="
    background-color: white;
    padding: 16px;
    border-left: 6px solid #811433;
    border-radius: 10px;
    margin-top: 10px;">
    <h3 style="color:#811433; margin-top:0;">Lecture analytique initiale</h3>
    <p style="color:#333; font-size:15px; line-height:1.6;">
        Cette étape permet de vérifier la qualité technique du jeu de données, de confirmer la cohérence
        des types de variables et d’enrichir la base avec de nouveaux indicateurs utiles à l’analyse :
        mois d’admission, tranche d’âge, coût journalier et variable binaire de séjour long.
        Ces transformations serviront directement à l’exploration visuelle et aux modèles prédictifs.
    </p>
</div>

<div style="background-color:#811433; padding:20px; border-radius:10px; margin:20px 0;">
    
<h1 style="color:white; text-align:center; margin:0;">
PARTIE 1 — EXPLORATION VISUELLE DES DONNÉES
</h1>

<p style="color:white; text-align:center; margin-top:10px;">
Analyse descriptive des patients, des hospitalisations, des durées et des coûts.
</p>

</div>

<div style="
    background-color:white;
    padding:18px;
    border-left:6px solid #811433;
    border-radius:10px;
    margin-top:10px;">
    <h2 style="color:#811433;">Distribution de l’âge des patients</h2>
    <p>Analyse de la structure démographique des patients hospitalisés.</p>
</div>

In [6]:
#1. Distribution de l’âge

data["Age"].describe().reset_index()

,index,Age
0,count,500.00
1,mean,47.18
2,std,25.05
3,min,1.00
4,25%,27.00
5,50%,49.00
6,75%,69.00
7,max,90.00


In [7]:
data["Age"].value_counts().sort_index().reset_index()

,Age,count
0,1,5
1,2,2
2,3,5
3,4,5
4,5,4
5,6,5
6,7,8
7,8,3
8,9,4
9,10,5


In [8]:
import plotly.express as px

fig = px.histogram(
    data,
    x="Age",
    nbins=20,
    color_discrete_sequence=["#811433"],
    title="Distribution de l'âge des patients"
)

fig.update_layout(
    title_font_color="#811433",
    xaxis_title="Âge",
    yaxis_title="Nombre de patients",
    template="plotly_white",
    bargap=0.2  # 🔥 espace entre les barres
)

fig.show()

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

L’âge moyen des patients est de <b>47 ans</b>, tandis que l’âge médian est de <b>49 ans</b>.
La proximité entre la moyenne et la médiane indique une distribution relativement équilibrée,
sans forte asymétrie.<br><br>

La population hospitalisée est très diversifiée, avec des âges allant de <b>1 à 90 ans</b>.
Cependant, 50% des patients ont un âge compris entre <b>27 et 69 ans</b>, ce qui montre
une forte concentration sur la population adulte.<br><br>

<b>Insight :</b><br>
L’hôpital prend en charge une large diversité de patients, mais la majorité des cas concerne
des adultes, ce qui peut orienter les besoins en équipements, en spécialités médicales
et en organisation des services.

</div>

In [9]:
# 2. Répartition par sexe
data["Sexe"].value_counts(normalize=True) * 100

Sexe
M   50.40
F   49.60
Name: proportion, dtype: float64

In [10]:


import plotly.express as px

fig = px.pie(
    data,
    names="Sexe",
    title="Répartition des patients par sexe",
    color="Sexe",
    color_discrete_map={
        "M": "#811433",
        "F": "#d9a5b3"
    }
)

# Affichage des pourcentages + labels
fig.update_traces(
    textinfo="percent+label",
    textfont_size=14
)

# Style global
fig.update_layout(
    title_font_color="#811433",
    template="plotly_white"
)

fig.show()

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

La répartition des patients par sexe est quasiment équilibrée, avec <b>50,4% d’hommes</b>
contre <b>49,6% de femmes</b>. Cette faible différence indique une absence de biais significatif
dans la fréquentation de l’hôpital selon le sexe.<br><br>

Cela suggère que les services hospitaliers sont sollicités de manière relativement homogène
par les deux sexes, ce qui renforce la représentativité des analyses globales réalisées
sur l’ensemble du jeu de données.<br><br>

<b>Insight :</b><br>
L’équilibre entre hommes et femmes permet d’explorer plus finement les différences éventuelles
en termes de maladies, de durée de séjour ou de coûts, sans risque de biais structurel lié au sexe.

</div>

In [11]:
# 3. Départements les plus sollicités
data["Departement"].value_counts().reset_index()

,Departement,count
0,Oncologie,90
1,Gériatrie,77
2,Orthopédie,76
3,Cardiologie,70
4,Dermatologie,68
5,Pneumologie,65
6,Neurologie,54


In [12]:
import plotly.express as px

# Préparer les données
dept_counts = data["Departement"].value_counts().reset_index()
dept_counts.columns = ["Departement", "Nombre"]

fig = px.bar(
    dept_counts,
    x="Nombre",
    y="Departement",
    orientation="h",
    title="Départements les plus sollicités",
    color_discrete_sequence=["#811433"],
    text="Nombre"  # 🔥 afficher les valeurs
)

# Style
fig.update_layout(
    title_font_color="#811433",
    xaxis_title="Nombre de patients",
    yaxis_title="Département",
    template="plotly_white"
)

# Position du texte
fig.update_traces(
    textposition="outside"
)

fig.show()

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

L’analyse des départements les plus sollicités montre une forte concentration des patients
dans certains services clés. L’<b’oncologie</b> apparaît comme le département le plus fréquenté,
suivi de la <b>gériatrie</b> et de l’<b’orthopédie</b>.<br><br>

Cette hiérarchie suggère une forte prévalence de pathologies lourdes et chroniques,
notamment liées au vieillissement de la population (gériatrie) ainsi qu’aux maladies graves
nécessitant des prises en charge prolongées (oncologie).<br><br>

Les départements comme la cardiologie, la dermatologie et la pneumologie présentent également
une activité importante, traduisant une diversité des besoins médicaux pris en charge par l’hôpital.<br><br>

<b>Insight :</b><br>
La forte sollicitation de certains départements peut entraîner une pression sur les ressources
(humaines, matérielles et financières). Cela peut nécessiter une optimisation de l’allocation
des ressources ou une priorisation stratégique de certains services.

</div>

In [13]:
# 4. Maladies les plus fréquentes
data["Maladie"].value_counts()

plt.show()

In [14]:
import plotly.express as px

# Préparer les données
maladie_counts = data["Maladie"].value_counts().reset_index()
maladie_counts.columns = ["Maladie", "Nombre"]

# Trier pour un affichage propre
maladie_counts = maladie_counts.sort_values(by="Nombre", ascending=True)

fig = px.bar(
    maladie_counts,
    x="Nombre",
    y="Maladie",
    orientation="h",
    title="Maladies les plus fréquentes",
    color_discrete_sequence=["#811433"],
    text="Nombre"
)

fig.update_layout(
    title_font_color="#811433",
    xaxis_title="Nombre de cas",
    yaxis_title="Maladie",
    template="plotly_white"
)

fig.update_traces(
    textposition="outside",
    marker_line_color="white",
    marker_line_width=1.5
)

fig.show()

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

L’analyse des maladies les plus fréquentes révèle que l’<b>eczéma</b> constitue la pathologie
la plus observée, suivi des <b>fractures</b> et du <b>cancer</b>.<br><br>

Cette distribution met en évidence une double réalité : d’une part, la forte présence
de pathologies courantes et généralement moins graves (comme l’eczéma), et d’autre part,
des maladies lourdes nécessitant une prise en charge complexe et prolongée
(comme le cancer et les infarctus).<br><br>

La présence significative de maladies chroniques telles que l’<b>hypertension</b> et
l’<b>Alzheimer</b> indique également un besoin important en suivi médical de long terme,
notamment pour les patients âgés.<br><br>

<b>Insight :</b><br>
La diversité des pathologies prises en charge montre que l’hôpital doit concilier
des soins de routine à forte fréquence avec des traitements spécialisés plus coûteux
et plus exigeants en ressources.

</div>

In [15]:
# 5. Distribution de la durée de séjour
data["DureeSejour"].describe().reset_index()

,index,DureeSejour
0,count,500.00
1,mean,7.95
2,std,4.21
3,min,1.00
4,25%,5.00
5,50%,8.00
6,75%,12.00
7,max,15.00


In [16]:
import plotly.express as px

fig = px.histogram(
    data,
    x="DureeSejour",
    nbins=20,
    title="Distribution de la durée de séjour",
    color_discrete_sequence=["#811433"]
)

fig.update_layout(
    title_font_color="#811433",
    xaxis_title="Durée de séjour (jours)",
    yaxis_title="Nombre de patients",
    template="plotly_white",
    bargap=0.2
)

fig.update_traces(
    marker_line_color="white",
    marker_line_width=1.5
)

fig.show()

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

La distribution de la durée de séjour montre une répartition relativement étalée,
avec des séjours allant de quelques jours à environ deux semaines.<br><br>

La majorité des patients semble se concentrer autour des durées moyennes,
ce qui traduit une certaine homogénéité dans la prise en charge des patients.
Cependant, la présence de séjours plus longs indique l’existence de cas
plus complexes nécessitant un suivi prolongé.<br><br>

<b>Insight :</b><br>
Les séjours longs représentent un enjeu majeur pour l’hôpital, car ils mobilisent
davantage de ressources (lits, personnel, équipements) et peuvent impacter
la capacité d’accueil globale.<br><br>

<b>Lecture stratégique :</b><br>
Une gestion optimisée de la durée de séjour est essentielle pour améliorer
l’efficacité opérationnelle et réduire les coûts hospitaliers.

</div>

In [17]:
# 6. Distribution des coûts

data["Cout"].describe().reset_index()

,index,Cout
0,count,500.00
1,mean,"4,060.12"
2,std,"2,418.81"
3,min,303.00
4,25%,"2,136.25"
5,50%,"3,769.50"
6,75%,"5,749.50"
7,max,"10,380.00"


In [18]:
import plotly.express as px

fig = px.histogram(
    data,
    x="Cout",
    nbins=20,
    title="Distribution des coûts d'hospitalisation",
    color_discrete_sequence=["#811433"]
)

fig.update_layout(
    title_font_color="#811433",
    xaxis_title="Coût (FCFA)",
    yaxis_title="Nombre de patients",
    template="plotly_white",
    bargap=0.2
)

fig.update_traces(
    marker_line_color="white",
    marker_line_width=1.5
)

fig.show()

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

La distribution des coûts d’hospitalisation présente une forte dispersion,
avec une concentration des patients autour des coûts moyens, mais également
la présence de coûts élevés atteignant des valeurs significatives.<br><br>

La forme de la distribution suggère une asymétrie vers la droite,
indiquant que quelques cas présentent des coûts particulièrement élevés
par rapport à la majorité des patients.<br><br>

<b>Insight clé :</b><br>
Ces coûts élevés sont probablement liés à des cas complexes nécessitant
des traitements spécialisés ou des durées de séjour prolongées.<br><br>

<b>Lecture stratégique :</b><br>
L’identification des facteurs expliquant ces coûts élevés est essentielle
pour optimiser la gestion financière de l’hôpital et améliorer l’efficacité
des soins.

</div>

<div style="background-color:#811433; padding:20px; border-radius:10px; margin:20px 0;">
    
<h1 style="color:white; text-align:center; margin:0;">
PARTIE 2 — ANALYSE DES HOSPITALISATIONS
</h1>

<p style="color:white; text-align:center; margin-top:10px;">
Analyse approfondie des durées de séjour, des départements et des facteurs influençant les hospitalisations.
</p>

</div>

In [21]:
# 1. Durée moyenne de séjour
data["DureeSejour"].mean()

np.float64(7.954)

In [23]:
data["DureeSejour"].describe().reset_index()

,index,DureeSejour
0,count,500.00
1,mean,7.95
2,std,4.21
3,min,1.00
4,25%,5.00
5,50%,8.00
6,75%,12.00
7,max,15.00


In [24]:
import plotly.express as px

fig = px.box(
    data,
    y="DureeSejour",
    title="Distribution de la durée de séjour",
    color_discrete_sequence=["#811433"]
)

fig.update_layout(
    title_font_color="#811433",
    template="plotly_white"
)

fig.show()

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

La durée moyenne de séjour est de <b>7,95 jours</b>, ce qui indique que les patients
restent en moyenne environ une semaine à l’hôpital.<br><br>

L’analyse du boxplot montre que la médiane se situe autour de <b>8 jours</b>,
confirmant une distribution relativement symétrique des durées de séjour.<br><br>

La majorité des patients (50%) a une durée de séjour comprise approximativement
entre <b>5 et 12 jours</b>, ce qui traduit une certaine homogénéité dans la prise en charge
des cas hospitaliers.<br><br>

Cependant, la présence de valeurs plus faibles (autour de 1–2 jours) et plus élevées
(jusqu’à environ 15 jours) indique une variabilité des cas, allant de situations simples
à des pathologies nécessitant un suivi plus long.<br><br>

<b>Insight :</b><br>
La relative concentration des durées autour de la moyenne suggère une gestion
globalement maîtrisée des hospitalisations. Toutefois, les séjours prolongés
représentent un enjeu important en termes de mobilisation des ressources
et d’optimisation de la capacité d’accueil.

</div>

In [26]:
# 2. Durée de séjour par département
data.groupby("Departement")["DureeSejour"].mean().sort_values(ascending=False).reset_index()

,Departement,DureeSejour
0,Orthopédie,8.30
1,Oncologie,8.08
2,Gériatrie,8.05
3,Neurologie,8.04
4,Pneumologie,7.82
5,Dermatologie,7.72
6,Cardiologie,7.60


In [27]:
dept_duree = data.groupby("Departement")["DureeSejour"].mean().reset_index()

fig = px.bar(
    dept_duree,
    x="DureeSejour",
    y="Departement",
    orientation="h",
    title="Durée moyenne de séjour par département",
    color_discrete_sequence=["#811433"],
    text="DureeSejour"
)

fig.update_layout(
    title_font_color="#811433",
    template="plotly_white"
)

fig.show()

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

L’analyse de la durée moyenne de séjour par département montre des variations relativement
faibles entre les différents services, avec des durées globalement comprises entre
<b>7,6 et 8,3 jours</b>.<br><br>

Le département d’<b>orthopédie</b> présente la durée moyenne de séjour la plus élevée
(environ <b>8,3 jours</b>), ce qui peut s’expliquer par des pathologies nécessitant
une récupération physique et un suivi prolongé (fractures, interventions chirurgicales).<br><br>

À l’inverse, la <b>cardiologie</b> affiche la durée moyenne la plus faible
(environ <b>7,6 jours</b>), suggérant une prise en charge plus rapide ou mieux standardisée.<br><br>

Les autres départements (oncologie, neurologie, gériatrie, pneumologie, dermatologie)
présentent des durées très proches de la moyenne globale, traduisant une certaine homogénéité
dans les pratiques de prise en charge.<br><br>

<b>Insight :</b><br>
Les écarts relativement faibles entre départements suggèrent une gestion cohérente des durées
de séjour à l’échelle de l’hôpital. Toutefois, les services comme l’orthopédie peuvent
représenter des points d’attention en raison de leur tendance à mobiliser les ressources
sur une durée plus longue.

</div>

In [28]:
# 3. Maladies associées aux longs séjours
## Création variable ##
seuil = data["DureeSejour"].mean()

data["SejourLong"] = data["DureeSejour"] > seuil

In [30]:
data[data["SejourLong"] == True]["Maladie"].value_counts().reset_index()

,Maladie,count
0,Eczéma,46
1,Fracture,42
2,Alzheimer,40
3,Cancer,39
4,Infarctus,36
5,Pneumonie,30
6,Hypertension,28


In [31]:
long_sejour = data[data["SejourLong"] == True]["Maladie"].value_counts().reset_index()
long_sejour.columns = ["Maladie", "Nombre"]

fig = px.bar(
    long_sejour,
    x="Nombre",
    y="Maladie",
    orientation="h",
    title="Maladies associées aux longs séjours",
    color_discrete_sequence=["#811433"],
    text="Nombre"
)

fig.update_layout(
    title_font_color="#811433",
    template="plotly_white"
)

fig.show()

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

L’analyse des maladies associées aux longs séjours met en évidence une concentration
de certains types de pathologies. L’<b>eczéma</b> apparaît comme la maladie la plus
représentée parmi les séjours prolongés (<b>46 cas</b>), suivie des <b>fractures</b> (<b>42 cas</b>)
et de la maladie d’<b>Alzheimer</b> (<b>40 cas</b>).<br><br>

D’autres pathologies comme le <b>cancer</b> (<b>39 cas</b>) et les <b>infarctus</b> (<b>36 cas</b>)
présentent également une forte présence, ce qui reflète la complexité et la gravité
de ces situations médicales nécessitant un suivi prolongé.<br><br>

Les maladies comme la <b>pneumonie</b> et l’<b>hypertension</b> apparaissent moins fréquentes
dans les longs séjours, suggérant une prise en charge plus rapide ou des protocoles
de traitement plus efficaces.<br><br>

<b>Insight :</b><br>
Les pathologies dominantes dans les longs séjours constituent des facteurs critiques
de consommation des ressources hospitalières. Leur identification permet de cibler
des actions d’optimisation, notamment en termes de protocoles de soins et de gestion
des durées d’hospitalisation.

</div>

In [32]:
# 4. Relation âge / durée de séjour
fig = px.scatter(
    data,
    x="Age",
    y="DureeSejour",
    title="Relation entre l'âge et la durée de séjour",
    color_discrete_sequence=["#811433"]
)

fig.update_layout(
    title_font_color="#811433",
    template="plotly_white"
)

fig.show()

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

L’analyse du nuage de points représentant la relation entre l’âge et la durée de séjour
ne met pas en évidence de corrélation linéaire claire entre ces deux variables.<br><br>

Les durées de séjour semblent réparties de manière relativement homogène
sur l’ensemble des tranches d’âge, ce qui suggère que l’âge, pris isolément,
n’est pas un facteur déterminant de la durée d’hospitalisation.<br><br>

On observe néanmoins une dispersion importante des durées pour toutes les catégories d’âge,
indiquant que d’autres facteurs (nature de la maladie, département ou type de traitement)
jouent un rôle plus significatif dans l’allongement des séjours.<br><br>

<b>Insight :</b><br>
La durée de séjour dépend davantage de facteurs cliniques et organisationnels
que de l’âge seul, ce qui souligne la nécessité d’une analyse multivariée
pour mieux comprendre les déterminants des hospitalisations prolongées.

</div>

In [35]:
# 5. Cohérence des dates
data["DureeCalculee"] = (data["DateSortie"] - data["DateAdmission"]).dt.days

data["Ecart"] = data["DureeSejour"] - data["DureeCalculee"]

data["Ecart"].value_counts()

Ecart
0    500
Name: count, dtype: int64

In [36]:
import plotly.express as px

ecart_counts = data["Ecart"].value_counts().reset_index()
ecart_counts.columns = ["Ecart", "Nombre"]

fig = px.bar(
    ecart_counts,
    x="Ecart",
    y="Nombre",
    title="Vérification de la cohérence des durées",
    color_discrete_sequence=["#811433"],
    text="Nombre"
)

fig.update_layout(
    title_font_color="#811433",
    template="plotly_white"
)

fig.show()

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

La vérification de la cohérence entre la durée de séjour déclarée et la durée calculée
à partir des dates d’admission et de sortie montre qu’il n’existe aucun écart
dans les données.<br><br>

En effet, l’ensemble des <b>500 observations</b> présente un écart nul,
ce qui confirme une parfaite cohérence des informations temporelles.<br><br>

<b>Insight :</b><br>
Cette qualité de données garantit la fiabilité des analyses réalisées par la suite,
notamment celles portant sur la durée de séjour et les coûts hospitaliers.<br><br>

<b>Lecture stratégique :</b><br>
Des données cohérentes constituent un prérequis essentiel pour toute analyse prédictive
ou décisionnelle, renforçant ainsi la crédibilité des résultats obtenus.

</div>

<div style="background-color:#811433; padding:20px; border-radius:10px; margin:20px 0;">
    
<h1 style="color:white; text-align:center; margin:0;">
PARTIE 3 — ANALYSE DES COÛTS
</h1>

<p style="color:white; text-align:center; margin-top:10px;">
Analyse des facteurs influençant les coûts hospitaliers et identification des leviers d’optimisation.
</p>

</div>

In [38]:
# 1. Coût moyen
data["Cout"].describe().reset_index()

,index,Cout
0,count,500.00
1,mean,"4,060.12"
2,std,"2,418.81"
3,min,303.00
4,25%,"2,136.25"
5,50%,"3,769.50"
6,75%,"5,749.50"
7,max,"10,380.00"


In [40]:
data["Cout"].mean()

np.float64(4060.118)

In [42]:
import plotly.graph_objects as go

cout_moyen = data["Cout"].mean()

fig = go.Figure(go.Indicator(
    mode="number",
    value=cout_moyen,
    title={"text": "Coût moyen d'hospitalisation"},
    number={"valueformat": ",.0f"}
))

fig.update_layout(
    paper_bgcolor="white",
    font_color="#811433"
)

fig.show()

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

Le coût moyen d’hospitalisation s’élève à <b>4,060</b>, avec une dispersion importante
entre les différents patients.<br><br>

La distribution des coûts met en évidence la présence de valeurs élevées,
indiquant que certains cas représentent une charge financière significative.<br><br>

<b>Insight :</b><br>
Une petite proportion de patients peut représenter une part importante
des coûts totaux de l’hôpital.

</div>

In [44]:
# 2. Coût par département
data.groupby("Departement")["Cout"].mean().sort_values(ascending=False).reset_index()

,Departement,Cout
0,Oncologie,"4,306.02"
1,Gériatrie,"4,222.49"
2,Orthopédie,"4,070.57"
3,Neurologie,"3,995.07"
4,Pneumologie,"3,987.02"
5,Dermatologie,"3,884.26"
6,Cardiologie,"3,842.89"


In [45]:
dept_cost = data.groupby("Departement")["Cout"].mean().reset_index()

fig = px.bar(
    dept_cost,
    x="Cout",
    y="Departement",
    orientation="h",
    title="Coût moyen par département",
    color_discrete_sequence=["#811433"],
    text="Cout"
)

fig.update_layout(
    title_font_color="#811433",
    template="plotly_white"
)

fig.show()

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>
Certains départements présentent des coûts moyens plus élevés,
notamment Oncologie et Gériatrie. Cela peut être lié à la complexité des soins
ou aux équipements nécessaires.<br><br>

<b>Insight :</b><br>
Les départements les plus coûteux doivent être analysés en priorité
pour identifier des opportunités d’optimisation.
</div>

In [47]:
# 3. Maladies les plus coûteuses
data.groupby("Maladie")["Cout"].mean().sort_values(ascending=False).reset_index()

,Maladie,Cout
0,Alzheimer,"4,320.91"
1,Infarctus,"4,234.06"
2,Fracture,"4,207.15"
3,Pneumonie,"4,143.42"
4,Cancer,"3,904.53"
5,Eczéma,"3,862.99"
6,Hypertension,"3,767.54"


In [48]:
maladie_cost = data.groupby("Maladie")["Cout"].mean().reset_index()

fig = px.bar(
    maladie_cost,
    x="Cout",
    y="Maladie",
    orientation="h",
    title="Coût moyen par maladie",
    color_discrete_sequence=["#811433"],
    text="Cout"
)

fig.update_layout(
    title_font_color="#811433",
    template="plotly_white"
)

fig.show()

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">
<b>Analyse :</b><br>
Certaines maladies présentent des coûts moyens significativement plus élevés,
notamment Alzheimer, Infarctus et Fracture, traduisant des traitements plus complexes.<br><br>

<b>Insight :</b><br>
Les maladies les plus coûteuses constituent un levier prioritaire
pour la gestion financière de l’hôpital.
</div>

In [49]:
# 4. Relation durée / coût
fig = px.scatter(
    data,
    x="DureeSejour",
    y="Cout",
    title="Relation entre durée de séjour et coût",
    color_discrete_sequence=["#811433"]
)

fig.update_layout(
    title_font_color="#811433",
    template="plotly_white"
)

fig.show()

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>
Une relation positive entre la durée de séjour et le coût est généralement observée,
indiquant que les séjours plus longs entraînent des coûts plus élevés. Plus le séjour est long plus le coût devient plus élevé.<br><br>

<b>Insight clé :</b><br>
La durée de séjour apparaît comme un facteur déterminant des coûts hospitaliers,
ce qui en fait un levier stratégique majeur pour l’optimisation financière.
</div>

In [51]:
# 5. Coût par traitement
data.groupby("Traitement")["Cout"].mean().sort_values(ascending=False).reset_index()

,Traitement,Cout
0,Radiothérapie,"4,380.97"
1,Soins spéciaux,"4,238.65"
2,Chirurgie,"4,213.94"
3,Physiothérapie,"3,997.98"
4,Médication,"3,852.49"
5,Antibiotiques,"3,674.28"


In [52]:
trait_cost = data.groupby("Traitement")["Cout"].mean().reset_index()

fig = px.bar(
    trait_cost,
    x="Cout",
    y="Traitement",
    orientation="h",
    title="Coût moyen par type de traitement",
    color_discrete_sequence=["#811433"],
    text="Cout"
)

fig.update_layout(
    title_font_color="#811433",
    template="plotly_white"
)

fig.show()

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>
Les coûts varient selon le type de traitement, certains étant plus coûteux
en raison de leur complexité ou de leur durée. Ici Radiothérapie, Soins Spéciaux et Chirurgie sont plus coûteux.<br><br>

<b>Insight :</b><br>
L’optimisation des protocoles de traitement peut contribuer
à réduire les coûts sans compromettre la qualité des soins.
</div>

In [53]:
# 6. COÛT PAR JOUR
data["CoutParJour"] = data["Cout"] / data["DureeSejour"]

In [54]:
fig = px.box(
    data,
    x="Departement",
    y="CoutParJour",
    title="Coût journalier par département",
    color_discrete_sequence=["#811433"]
)

fig.update_layout(title_font_color="#811433")
fig.show()


<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">
<b>Analyse :</b><br>
Le coût journalier permet de distinguer les services intrinsèquement coûteux
de ceux dont le coût est simplement lié à la durée de séjour.<br><br>

<b>Insight :</b><br>
Un département avec un coût journalier élevé est structurellement plus cher,
indépendamment de la durée des hospitalisations.
</div>

In [55]:
# 7. COÛT VS DURÉE
fig = px.scatter(
    data,
    x="DureeSejour",
    y="Cout",
    color="Departement",
    title="Relation coût vs durée de séjour"
)

fig.update_layout(title_font_color="#811433")
fig.show()

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>
La relation entre durée et coût montre une tendance globale croissante,
mais avec une dispersion importante.<br><br>

<b>Insight :</b><br>
Cela signifie que la durée explique une partie du coût, mais que d’autres facteurs
(complexité médicale, traitement) jouent également un rôle déterminant.
</div>

In [56]:
# 8. DÉTECTION DES CAS EXTRÊMES
data.sort_values(by="Cout", ascending=False).head(10)

,PatientID,Age,Sexe,Departement,Maladie,DureeSejour,Cout,DateAdmission,DateSortie,Traitement,DureeCalculee,EcartDuree,MoisAdmission,NomMoisAdmission,JourAdmission,TrancheAge,CoutParJour,SejourLong,Ecart
282,283,65,M,Oncologie,Pneumonie,15,10380,2025-09-08,2025-09-23,Radiothérapie,15,0,9,September,Monday,51-65,692.00,True,0
182,183,49,F,Oncologie,Cancer,15,10320,2025-02-08,2025-02-23,Soins spéciaux,15,0,2,February,Saturday,36-50,688.00,True,0
426,427,65,F,Orthopédie,Eczéma,15,10110,2025-12-01,2025-12-16,Radiothérapie,15,0,12,December,Monday,51-65,674.00,True,0
330,331,39,M,Oncologie,Cancer,15,9870,2025-10-17,2025-11-01,Physiothérapie,15,0,10,October,Friday,36-50,658.00,True,0
63,64,55,M,Cardiologie,Infarctus,15,9795,2024-12-23,2025-01-07,Chirurgie,15,0,12,December,Monday,51-65,653.00,True,0
14,15,72,M,Oncologie,Pneumonie,14,9702,2025-11-10,2025-11-24,Médication,14,0,11,November,Monday,66-80,693.00,True,0
160,161,64,F,Oncologie,Alzheimer,14,9646,2025-10-11,2025-10-25,Radiothérapie,14,0,10,October,Saturday,51-65,689.00,True,0
355,356,7,F,Gériatrie,Hypertension,15,9510,2025-06-08,2025-06-23,Physiothérapie,15,0,6,June,Sunday,0-18,634.00,True,0
253,254,65,F,Cardiologie,Alzheimer,14,9506,2025-08-14,2025-08-28,Médication,14,0,8,August,Thursday,51-65,679.00,True,0
122,123,69,F,Gériatrie,Alzheimer,14,9436,2025-03-17,2025-03-31,Radiothérapie,14,0,3,March,Monday,66-80,674.00,True,0


<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>
Les cas les plus coûteux représentent des situations atypiques
nécessitant une attention particulière.<br><br>

<b>Insight :</b><br>
Une petite proportion de patients peut générer une part disproportionnée
des coûts totaux.
</div>

<div style="background-color:#811433; padding:20px; border-radius:10px; margin:20px 0;">
    
<h1 style="color:white; text-align:center; margin:0;">
PARTIE 5 — MACHINE LEARNING
</h1>

<p style="color:white; text-align:center; margin-top:10px;">
Modélisation prédictive des coûts et des séjours hospitaliers.
</p>

</div>



<div style="
    background-color: white;
    padding: 14px 18px;
    border-radius: 10px;
    border: 2px solid #811433;
    margin: 18px 0 12px 0;">
    <h2 style="
        color: #811433;
        margin: 0;
        font-family: Arial, sans-serif;">
        OPTION 1 — RÉGRESSION
    </h2>
</div>


In [69]:
# 1. Préparation des données
features = ["Age", "Sexe", "Departement", "Maladie", "Traitement", "DureeSejour"]

X = data[features]
y = data["Cout"]

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)
display(X.head())
display(y.head())

Dimensions de X : (500, 6)
Dimensions de y : (500,)


,Age,Sexe,Departement,Maladie,Traitement,DureeSejour
0,82,M,Cardiologie,Cancer,Chirurgie,5
1,18,M,Oncologie,Cancer,Médication,15
2,76,F,Cardiologie,Infarctus,Chirurgie,2
3,65,M,Neurologie,Fracture,Physiothérapie,12
4,70,F,Orthopédie,Alzheimer,Médication,10


0    2125
1    8685
2     822
3    7584
4    4420
Name: Cout, dtype: int64

In [70]:
# 2. Train / Test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (400, 6)
X_test  : (100, 6)
y_train : (400,)
y_test  : (100,)


In [71]:
# 3. Pipeline de preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = ["Age", "DureeSejour"]
cat_cols = ["Sexe", "Departement", "Maladie", "Traitement"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

print("Variables numériques :", num_cols)
print("Variables catégorielles :", cat_cols)
display(preprocessor)

Variables numériques : ['Age', 'DureeSejour']
Variables catégorielles : ['Sexe', 'Departement', 'Maladie', 'Traitement']


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [72]:
# 4. Modèles
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    "Linear Regression": Pipeline([
        ("prep", preprocessor),
        ("model", LinearRegression())
    ]),
    
    "Random Forest": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestRegressor(n_estimators=100, random_state=42))
    ]),
    
    "Gradient Boosting": Pipeline([
        ("prep", preprocessor),
        ("model", GradientBoostingRegressor())
    ])
}

for name, model in models.items():
    print(f"{name} : {model}")

Linear Regression : Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Age', 'DureeSejour']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Sexe', 'Departement',
                                                   'Maladie',
                                                   'Traitement'])])),
                ('model', LinearRegression())])
Random Forest : Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Age', 'DureeSejour']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Sexe', 'Depart

In [65]:
# 5. Entraînement + Évaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

results_reg = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    
    results_reg.append([name, mae, rmse, r2])

import pandas as pd
results_reg = pd.DataFrame(results_reg, columns=["Model", "MAE", "RMSE", "R2"])
results_reg

,Model,MAE,RMSE,R2
0,Linear Regression,792.10,"1,016.21",0.82
1,Random Forest,773.61,"1,029.66",0.81
2,Gradient Boosting,768.56,974.48,0.83


<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

Les performances des modèles de régression montrent que le <b>Gradient Boosting</b>
est le modèle le plus performant, avec un <b>R² de 0,83</b>, un <b>RMSE de 974,48</b>
et un <b>MAE de 768,56</b>.<br><br>

La <b>régression linéaire</b> présente également de bonnes performances (R² = 0,82),
ce qui indique que les relations entre les variables explicatives et le coût
sont en partie linéaires.<br><br>

Le modèle <b>Random Forest</b> obtient des performances légèrement inférieures
(R² = 0,81), ce qui suggère que, dans ce cas précis, les modèles d’ensemble
n’apportent pas un gain significatif par rapport aux modèles plus simples.<br><br>

<b>Insight :</b><br>
La performance globalement élevée des modèles (R² > 0,80) montre que les variables
disponibles expliquent une grande partie de la variation des coûts hospitaliers.<br><br>

<b>Lecture stratégique :</b><br>
Le modèle Gradient Boosting est le plus adapté pour prédire les coûts,
car il capture mieux les relations complexes tout en conservant une bonne précision.

<b>Comparaison des modèles :</b><br>
Bien que les trois modèles présentent des performances proches,
le Gradient Boosting offre le meilleur compromis entre précision et robustesse,
ce qui en fait le modèle retenu pour la suite de l’analyse.

</div>

In [67]:
# 6. Importance des variables
# Récupérer le modèle entraîné
model_best = models["Random Forest"]
model_best.fit(X_train, y_train)

# Récupérer les importances
importances = model_best.named_steps["model"].feature_importances_

# Récupérer les noms des variables après transformation
feature_names = model_best.named_steps["prep"].get_feature_names_out()

# Créer un DataFrame propre
import pandas as pd

feat_imp = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

# Trier
feat_imp = feat_imp.sort_values(by="Importance", ascending=False)

feat_imp.head(10)

,Feature,Importance
1,num__DureeSejour,0.83
0,num__Age,0.05
18,cat__Traitement_Antibiotiques,0.01
12,cat__Maladie_Cancer,0.01
23,cat__Traitement_Soins spéciaux,0.01
7,cat__Departement_Neurologie,0.01
8,cat__Departement_Oncologie,0.01
21,cat__Traitement_Physiothérapie,0.01
19,cat__Traitement_Chirurgie,0.01
15,cat__Maladie_Hypertension,0.01


In [68]:
import plotly.express as px

top_features = feat_imp.head(10)

fig = px.bar(
    top_features,
    x="Importance",
    y="Feature",
    orientation="h",
    title="Importance des variables dans la prédiction du coût",
    color_discrete_sequence=["#811433"],
    text="Importance"
)

fig.update_layout(
    title_font_color="#811433",
    template="plotly_white"
)

fig.show()

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

L’analyse de l’importance des variables montre clairement que la <b>durée de séjour</b>
(<b>DureeSejour</b>) est de loin le facteur le plus déterminant dans la prédiction
des coûts hospitaliers, avec une importance d’environ <b>0,83</b>.<br><br>

Ce résultat indique que la majeure partie de la variation des coûts s’explique directement
par la durée d’hospitalisation, confirmant les analyses exploratoires réalisées précédemment.<br><br>

L’<b>âge</b> (<b>Age</b>) apparaît comme un facteur secondaire, avec une importance
nettement plus faible (environ <b>0,05</b>), ce qui montre qu’il influence les coûts,
mais de manière limitée.<br><br>

Les autres variables (maladie, département, traitement) ont une contribution marginale,
chacune ayant une importance inférieure à <b>0,01</b>. Cela suggère que leur effet
sur le coût est indirect ou diffus.<br><br>

<b>Insight clé :</b><br>
Le coût hospitalier est principalement piloté par la durée de séjour,
tandis que les autres variables jouent un rôle d’ajustement plus fin.<br><br>

<b>Lecture stratégique :</b><br>
La réduction des durées de séjour constitue le levier le plus efficace
pour maîtriser les coûts hospitaliers. Les efforts d’optimisation doivent donc
se concentrer prioritairement sur l’amélioration des parcours de soins
et la gestion des hospitalisations prolongées.

</div>



<div style="
    background-color: white;
    padding: 14px 18px;
    border-radius: 10px;
    border: 2px solid #811433;
    margin: 18px 0 12px 0;">
    <h2 style="
        color: #811433;
        margin: 0;
        font-family: Arial, sans-serif;">
        OPTION 2 — CLASSIFICATION
    </h2>
</div>


In [85]:
# 1. Création variable cible
seuil = data["DureeSejour"].mean()
data["SejourLong"] = (data["DureeSejour"] > seuil).astype(int)
print("Seuil :", seuil)
print(data["SejourLong"].value_counts())
display(data[["DureeSejour", "SejourLong"]].head())

Seuil : 7.954
SejourLong
1    261
0    239
Name: count, dtype: int64


,DureeSejour,SejourLong
0,5,0
1,15,1
2,2,0
3,12,1
4,10,1


In [86]:
# 2. Données
features = ["Age", "Sexe", "Departement", "Maladie", "Traitement", "DureeSejour"]

X = data[features]
y = data["SejourLong"]
print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)
display(X.head())
display(y.head())

Dimensions de X : (500, 6)
Dimensions de y : (500,)


,Age,Sexe,Departement,Maladie,Traitement,DureeSejour
0,82,M,Cardiologie,Cancer,Chirurgie,5
1,18,M,Oncologie,Cancer,Médication,15
2,76,F,Cardiologie,Infarctus,Chirurgie,2
3,65,M,Neurologie,Fracture,Physiothérapie,12
4,70,F,Orthopédie,Alzheimer,Médication,10


0    0
1    1
2    0
3    1
4    1
Name: SejourLong, dtype: int64

In [87]:
# 3. Modèles
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

models_clf = {
    "Logistic Regression": Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    
    "Random Forest": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestClassifier())
    ]),
    
    "Decision Tree": Pipeline([
        ("prep", preprocessor),
        ("model", DecisionTreeClassifier())
    ])
}

for name, model in models_clf.items():
    print(name)
    print(model)
    print("-" * 50)

Logistic Regression
Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Age', 'DureeSejour']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Sexe', 'Departement',
                                                   'Maladie',
                                                   'Traitement'])])),
                ('model', LogisticRegression(max_iter=1000))])
--------------------------------------------------
Random Forest
Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Age', 'DureeSejour']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
  

In [82]:
#### refaire le train/test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_clf, y_test_clf = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [90]:
# 4. Évaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

results_clf = []

for name, model in models_clf.items():
    model.fit(X_train, y_train_clf)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test_clf, preds)
    prec = precision_score(y_test_clf, preds)
    rec = recall_score(y_test_clf, preds)
    f1 = f1_score(y_test_clf, preds)

    results_clf.append([name, acc, prec, rec, f1])

results_clf = pd.DataFrame(
    results_clf,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1"]
)

display(results_clf)

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,1.00,1.00,1.00,1.00
1,Random Forest,1.00,1.00,1.00,1.00
2,Decision Tree,1.00,1.00,1.00,1.00


<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

Les résultats de classification montrent des performances parfaites pour l’ensemble
des modèles testés (Logistic Regression, Random Forest et Decision Tree),
avec des scores de <b>100%</b> pour l’Accuracy, la Precision, le Recall et le F1-score.<br><br>

Ces résultats indiquent que les modèles sont capables de prédire parfaitement
les séjours longs à partir des variables disponibles.<br><br>

<b>Interprétation critique :</b><br>
Un tel niveau de performance est exceptionnel et peut s’expliquer par le fait
que la variable cible (<b>SejourLong</b>) est directement construite à partir
de la variable <b>DureeSejour</b>, qui est également utilisée comme variable explicative
dans le modèle.<br><br>

Cela signifie que le modèle “voit” déjà l’information qu’il doit prédire,
ce qui crée une situation de <b>fuite de données (data leakage)</b>.<br><br>

<b>Insight :</b><br>
Les modèles ne sont pas réellement prédictifs dans ce cas, car ils exploitent
une information directement liée à la cible. Pour une modélisation plus réaliste,
il serait nécessaire d’exclure la variable <b>DureeSejour</b> des variables explicatives.

“Les performances parfaites s’expliquent par une fuite d’information liée à la variable DureeSejour.”

</div>

In [84]:
# 5. Facteurs des séjours longs
model_clf_best = models_clf["Random Forest"]
model_clf_best.fit(X_train, y_train_clf)

model_clf_best.named_steps["model"].feature_importances_

array([0.06605217, 0.77439493, 0.00934389, 0.00741121, 0.00763408,
       0.00726939, 0.00697372, 0.00525619, 0.00818688, 0.00681686,
       0.00552053, 0.00791145, 0.00727155, 0.00775605, 0.0064852 ,
       0.00725125, 0.00797115, 0.00599449, 0.00764234, 0.00589974,
       0.00756837, 0.00801195, 0.00807206, 0.00730456])

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

L’analyse de l’importance des variables dans la prédiction des séjours longs
met en évidence une forte domination d’une seule variable, avec une importance
d’environ <b>77%</b>, ce qui indique qu’elle explique l’essentiel du modèle.<br><br>

Une deuxième variable présente une contribution plus modérée
(environ <b>6%</b>), tandis que l’ensemble des autres variables ont un impact
très faible (inférieur à <b>1%</b> chacune).<br><br>

<b>Interprétation :</b><br>
Ce résultat confirme que la prédiction des séjours longs repose principalement
sur une variable centrale, qui est la <b>durée de séjour</b> elle-même,
utilisée indirectement pour construire la variable cible.<br><br>

<b>Insight :</b><br>
Les autres variables (âge, maladie, département, traitement) jouent un rôle
secondaire et n’apportent qu’un ajustement marginal dans la prédiction.<br><br>

<b>Limite du modèle :</b><br>
La forte dominance d’une seule variable révèle une situation de <b>fuite de données (data leakage)</b>,
ce qui limite la capacité du modèle à généraliser sur de nouvelles données.

</div>

### REMARQUE
<b>Remarque méthodologique :</b><br>

Les performances parfaites obtenues dans les modèles de classification
s’expliquent par le fait que la variable cible (<b>SejourLong</b>)
est directement construite à partir de la variable <b>DureeSejour</b>,
qui est également utilisée comme variable explicative.<br><br>

Cette situation correspond à une <b>fuite de données (data leakage)</b>,
où le modèle dispose indirectement de l’information qu’il doit prédire.<br><br>

<b>Conséquence :</b><br>
Les performances obtenues sont artificiellement élevées et ne reflètent
pas la capacité réelle du modèle à généraliser sur de nouvelles données.<br><br>

<b>Amélioration possible :</b><br>
Pour une modélisation plus réaliste, il serait pertinent d’exclure
la variable <b>DureeSejour</b> des variables explicatives.

</div>

<div style="background-color:#811433; padding:20px; border-radius:10px; margin:20px 0;">
    
<h1 style="color:white; text-align:center; margin:0;">
PARTIE 6 — INTERPRÉTATION DES RÉSULTATS
</h1>

<p style="color:white; text-align:center; margin-top:10px;">
Analyse stratégique et recommandations basées sur les résultats obtenus.
</p>

</div>



<div style="
    background-color: white;
    padding: 14px 18px;
    border-radius: 10px;
    border: 2px solid #811433;
    margin: 18px 0 12px 0;">
    <h4 style="
        color: #811433;
        margin: 0;
        font-family: Arial, sans-serif;">
        1. Quels facteurs expliquent les coûts hospitaliers ?
    </h4>
</div>


<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

Les résultats montrent que la <b>durée de séjour</b> est de loin le facteur
le plus déterminant des coûts hospitaliers, avec une importance très élevée
dans les modèles de machine learning.<br><br>

L’<b>âge</b> intervient comme un facteur secondaire, tandis que les variables
liées à la <b>maladie</b>, au <b>département</b> et au <b>traitement</b>
ont une influence plus diffuse et indirecte.<br><br>

<b>Interprétation :</b><br>
Ces variables influencent principalement les coûts en agissant sur la durée
de séjour, qui joue un rôle central dans la structure des dépenses.<br><br>

<b>Insight :</b><br>
Le coût hospitalier est essentiellement piloté par le temps passé à l’hôpital,
ce qui en fait le principal levier d’optimisation financière.

</div>



<div style="
    background-color: white;
    padding: 14px 18px;
    border-radius: 10px;
    border: 2px solid #811433;
    margin: 18px 0 12px 0;">
    <h4 style="
        color: #811433;
        margin: 0;
        font-family: Arial, sans-serif;">
        2. Quels profils de patients ont les séjours les plus longs ?
    </h4>
</div>


<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

Les séjours les plus longs sont associés à certaines pathologies spécifiques,
notamment les maladies nécessitant des soins prolongés comme le <b>cancer</b>,
les <b>fractures</b> ou les maladies neurodégénératives telles que <b>Alzheimer</b>.<br><br>

Les départements comme l’<b>orthopédie</b> et l’<b>oncologie</b> concentrent
également des durées de séjour plus élevées, en raison de la complexité
des traitements et des phases de récupération.<br><br>

L’analyse a montré que l’<b>âge seul n’est pas un facteur déterminant</b>,
ce qui indique que les séjours longs sont davantage liés à des facteurs médicaux
qu’à des caractéristiques démographiques.

<b>Insight :</b><br>
Les profils à risque de séjours longs sont définis par la nature de la pathologie
et le type de prise en charge, plutôt que par l’âge ou le sexe.

</div>



<div style="
    background-color: white;
    padding: 14px 18px;
    border-radius: 10px;
    border: 2px solid #811433;
    margin: 18px 0 12px 0;">
    <h4 style="
        color: #811433;
        margin: 0;
        font-family: Arial, sans-serif;">
        3. Les modèles prédictifs sont-ils fiables ?
    </h4>
</div>


<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Analyse :</b><br>

Les modèles de régression présentent de bonnes performances (R² supérieur à 0,80),
ce qui indique une capacité élevée à expliquer les coûts hospitaliers à partir
des variables disponibles.<br><br>

En revanche, les modèles de classification affichent des performances parfaites,
ce qui s’explique par une situation de <b>fuite de données (data leakage)</b>,
liée à l’utilisation de la variable <b>DureeSejour</b> dans la construction
de la variable cible.<br><br>

<b>Interprétation :</b><br>
Les modèles de régression sont fiables et exploitables, tandis que les modèles
de classification nécessitent une amélioration pour être utilisés dans un contexte réel.

<b>Insight :</b><br>
La fiabilité des modèles dépend fortement de la qualité des variables et
de la rigueur méthodologique dans la construction du modèle.

</div>



<div style="
    background-color: white;
    padding: 14px 18px;
    border-radius: 10px;
    border: 2px solid #811433;
    margin: 18px 0 12px 0;">
    <h4 style="
        color: #811433;
        margin: 0;
        font-family: Arial, sans-serif;">
        4. Comment l’hôpital pourrait-il optimiser ses ressources ?
    </h4>
</div>


<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Recommandations stratégiques :</b><br>

L’analyse met en évidence plusieurs leviers d’optimisation :

<ul>
<li><b>Réduction des durées de séjour :</b> optimiser les protocoles de soins et améliorer la coordination entre services afin de limiter les hospitalisations prolongées.</li>

<li><b>Ciblage des pathologies coûteuses :</b> concentrer les efforts sur les maladies générant les séjours longs et les coûts élevés.</li>

<li><b>Optimisation des ressources par département :</b> ajuster les moyens humains et matériels en fonction de la charge et des coûts par service.</li>

<li><b>Utilisation des modèles prédictifs :</b> anticiper les patients à risque de séjours longs pour améliorer la planification hospitalière.</li>

</ul>

<b>Insight global :</b><br>
Une gestion basée sur les données permet d’améliorer à la fois l’efficacité opérationnelle
et la performance financière de l’hôpital.

</div>

<div style="background-color:#f9f9f9; padding:14px; border-radius:10px;">

<b>Conclusion générale :</b><br>

Ce travail a permis de mener une analyse complète des données hospitalières,
allant de l’exploration descriptive à la modélisation prédictive.<br><br>

Les résultats ont mis en évidence le rôle central de la <b>durée de séjour</b>
dans l’explication des coûts hospitaliers, ainsi que l’importance des facteurs
cliniques dans la gestion des hospitalisations prolongées.<br><br>

Les modèles de machine learning ont montré de bonnes performances,
confirmant la capacité à prédire les coûts de manière fiable,
tout en soulignant certaines limites méthodologiques dans la classification
des séjours longs.<br><br>

Enfin, cette analyse a permis de proposer des recommandations concrètes
pour optimiser l’utilisation des ressources hospitalières,
notamment à travers la réduction des durées de séjour et l’anticipation
des cas complexes.<br><br>

<b>Ouverture :</b><br>
L’intégration de données supplémentaires (gravité des cas, historique médical)
et l’amélioration des modèles permettraient d’affiner davantage les prédictions
et de renforcer la prise de décision.

</div>